# A/B Test 05 — Retention (and how to judge a non-significant result)

**Question.** Did the treatment affect 30-day retention?

**Metric type.** `returned_30d` is binary 0/1 → proportion → **two-proportion z-test**.

This notebook focuses on **judgement**: what to do when a result is *not* significant but the gap looks favorable.


In [1]:
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

df = pd.read_parquet("../data/ab_test_data.parquet")
df.head()

,user_id,group,clicked,purchased,order_value,returned_30d
0,T01163,treatment,1,0,0.00,1
1,T04385,treatment,0,0,0.00,1
2,C01902,control,1,1,96.31,0
3,C03397,control,0,0,0.00,0
4,T05695,treatment,1,0,0.00,1


In [2]:
# Data quality checks (run before trusting any metric)
print("shape:", df.shape)
print("\nnulls:\n", df.isnull().sum())
print("\nduplicate user_id:", df["user_id"].duplicated().sum())

# SRM (sample ratio mismatch): groups should be ~50/50
print("\ngroup sizes:\n", df["group"].value_counts())

# Binary columns must contain only 0/1
for col in ["clicked", "purchased", "returned_30d"]:
    print(col, "unique:", sorted(df[col].unique()))

shape: (12000, 6)

nulls:
 user_id         0
group           0
clicked         0
purchased       0
order_value     0
returned_30d    0
dtype: int64

duplicate user_id: 0

group sizes:
 group
treatment    6000
control      6000
Name: count, dtype: int64
clicked unique: [np.int64(0), np.int64(1)]
purchased unique: [np.int64(0), np.int64(1)]
returned_30d unique: [np.int64(0), np.int64(1)]


## Compute the metric

In [3]:
summary = df.groupby("group")["returned_30d"].agg(["sum", "count"])
summary["retention_rate"] = summary["sum"] / summary["count"]
print(summary)

            sum  count  retention_rate
group                                 
control    2318   6000        0.386333
treatment  2391   6000        0.398500


## Statistical test — two-proportion z-test

In [4]:
retained = summary["sum"].values
n        = summary["count"].values
stat, pval = proportions_ztest(retained, n)

print(f"control = {summary['retention_rate']['control']:.4f} | treatment = {summary['retention_rate']['treatment']:.4f}")
print(f"p-value = {pval:.4f}")
print("Significant" if pval < 0.05 else "Not significant")

control = 0.3863 | treatment = 0.3985
p-value = 0.1723
Not significant


## Result

Retention is **38.6% (control)** vs **39.9% (treatment)**, a +1.2 pp gap, p ≈ 0.17 → **not significant**.

A high p-value does **not** prove "no effect". It could be a true null, or a real-but-small effect the test wasn't powered to detect. The distance of p from 0.05 says nothing about power.

## Confidence interval — is it "no effect" or just undetectable?

When p > 0.05 **and** the observed gap is non-trivial and in a useful direction (as here), compute the confidence interval of the difference to tell the two cases apart.

In [5]:
count = summary["sum"].values
nobs  = summary["count"].values

# 95% CI for the difference (treatment - control)
low, high = confint_proportions_2indep(
    count[1], nobs[1],   # treatment
    count[0], nobs[0],   # control
    method="wald",
)
print(f"retention diff (treatment - control) 95% CI: [{low:.4f}, {high:.4f}]")

# Read it:
#  - contains 0   -> not significant (consistent with no effect)
#  - wide         -> underpowered / inconclusive (need more data)
#  - narrow & ~0  -> genuinely little/no effect

retention diff (treatment - control) 95% CI: [-0.0053, 0.0296]


## Concepts

### Confidence interval, in one table
| CI looks like | Meaning |
|---|---|
| Contains 0 | not significant |
| Narrow & around 0 | genuinely little/no effect |
| Wide | underpowered / inconclusive → need more data |
| Entirely above/below 0 | significant, and you can see the effect size |

Here the interval contains 0 and is fairly wide → the result is inconclusive; a small real effect could be hiding. The CI is more informative than the p-value alone because it shows **how big** the effect could plausibly be.

### Metric roles depend on the hypothesis
| Role | Meaning | Here |
|---|---|---|
| Primary | What the experiment tries to improve | conversion / ARPU |
| Downstream | The next step the primary feeds | CTR → conversion → revenue |
| Guardrail | Must not get worse | returns, page speed, **retention** |

This experiment is a banner/conversion test, so **retention is a guardrail**: not significant but not declining → it does not block the launch (but we cannot claim a retention gain). For a loyalty feature, retention would instead be the **primary** metric.

### Leading vs lagging metrics
Conversion is a **leading** metric (moves immediately); retention is **lagging** (reveals long-term health later). A short A/B test can miss a slow retention decay, so keep a **holdback group** and monitor retention/returns as ongoing guardrails after launch — the ship decision is "ship + monitor", not a one-time call.
